# ІС-21 Мамчик Денис ЛР6

In [11]:
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, array_to_img
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import os
import kagglehub
import shutil
from pathlib import Path
from tensorflow.keras.models import load_model
import cv2
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping
import random

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp -r /content/drive/MyDrive/toyota /content/

In [4]:
LOGO_DIR="/content/toyota"
AUGMENTED_DIR="/content/toyota_augmented"
TRAIN_DIR="/content/data_for_training"
MODEL_PATH="/content/toyota_xception_model.h5"

TOYOTA_DIR = os.path.join(TRAIN_DIR, "toyota")
RANDOM_DIR_DST = os.path.join(TRAIN_DIR, "random")

IMG_SIZE = (224, 224)
AUG_PER_IMAGE = 5
NUM_RANDOM_IMAGES = 800

os.makedirs(AUGMENTED_DIR, exist_ok=True)
os.makedirs(TOYOTA_DIR, exist_ok=True)
os.makedirs(RANDOM_DIR_DST, exist_ok=True)

In [5]:
path = kagglehub.dataset_download("pankajkumar2002/random-image-sample-dataset")
print("Path to dataset files:", path)

RANDOM_IMAGES_PATH = os.path.join(path, 'data')
print("Шлях до директорії з рандомними зображеннями:", RANDOM_IMAGES_PATH)
print("Вміст директорії з рандомними зображеннями:", os.listdir(RANDOM_IMAGES_PATH))

Path to dataset files: /kaggle/input/random-image-sample-dataset
Шлях до директорії з рандомними зображеннями: /kaggle/input/random-image-sample-dataset/data
Вміст директорії з рандомними зображеннями: ['22735.jpg', '22706.jpg', '20513.jpg', '20088.jpg', '22608.jpg', '24201.jpg', '23407.jpg', '23149.jpg', '21494.jpg', '20777.jpg', '23274.jpg', '23775.jpg', '22046.jpg', '20938.jpg', '23436.jpg', '21575.jpg', '23710.jpg', '20833.jpg', '20684.jpg', '20554.jpg', '23753.jpg', '23474.jpg', '23273.jpg', '23031.jpg', '23098.jpg', '20100.jpg', '22711.jpg', '20076.jpg', '23220.jpg', '23094.jpg', '21232.jpg', '21947.jpg', '21154.jpg', '23596.jpg', '20978.jpg', '20825.jpg', '21701.jpg', '24267.jpg', '20906.jpg', '21093.jpg', '23053.jpg', '20874.jpg', '21787.jpg', '23625.jpg', '22666.jpg', '22459.jpg', '20937.jpg', '20267.jpg', '20475.jpg', '21612.jpg', '22742.jpg', '21126.jpg', '24287.jpg', '20079.jpg', '20762.jpg', '23819.jpg', '22368.jpg', '23272.jpg', '24083.jpg', '21278.jpg', '23351.jpg', '211

In [6]:
# Копіювання перших NUM_RANDOM_IMAGES файлів до RANDOM_DIR_DST
random_files = [f for f in os.listdir(RANDOM_IMAGES_PATH) if os.path.isfile(os.path.join(RANDOM_IMAGES_PATH, f))]
random_files = random_files[:NUM_RANDOM_IMAGES]
for filename in random_files:
    src_path = os.path.join(RANDOM_IMAGES_PATH, filename)
    dst_path = os.path.join(RANDOM_DIR_DST, filename)
    shutil.copy(src_path, dst_path)
print(f"Скопійовано {len(random_files)} рандомних зображень до {RANDOM_DIR_DST}")

Скопійовано 800 рандомних зображень до /content/data_for_training/random


In [7]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=False,
    fill_mode='nearest'
)

for filename in os.listdir(LOGO_DIR):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(LOGO_DIR, filename)
        img = load_img(img_path, target_size=IMG_SIZE)
        x = img_to_array(img)
        x = np.expand_dims(x, axis=0)
        i = 0
        for batch in datagen.flow(x, batch_size=1,
                                  save_to_dir=AUGMENTED_DIR,
                                  save_prefix='aug',
                                  save_format='jpg'):
            i += 1
            if i >= AUG_PER_IMAGE:
                break

for filename in os.listdir(LOGO_DIR):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        src_path = os.path.join(LOGO_DIR, filename)
        dst_path = os.path.join(AUGMENTED_DIR, "orig_" + filename)
        shutil.copy(src_path, dst_path)

for fname in os.listdir(AUGMENTED_DIR):
    shutil.copy(os.path.join(AUGMENTED_DIR, fname), TOYOTA_DIR)

In [8]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    directory="/content/data_for_training",
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    directory="/content/data_for_training",
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print(train_generator.class_indices)

Found 1358 images belonging to 2 classes.
Found 339 images belonging to 2 classes.
{'random': 0, 'toyota': 1}


In [9]:
# Підрахунок кількості зображень у кожному класі для навчального набору
train_classes = train_generator.classes
unique_train_classes, counts_train = np.unique(train_classes, return_counts=True)
class_names_train = {v: k for k, v in train_generator.class_indices.items()}
print("\nРозподіл класів у навчальному наборі:")
for i in range(len(unique_train_classes)):
    print(f"Клас {class_names_train[unique_train_classes[i]]}: {counts_train[i]} зображень")

# Підрахунок кількості зображень у кожному класі для валідаційного набору
val_classes = val_generator.classes
unique_val_classes, counts_val = np.unique(val_classes, return_counts=True)
class_names_val = {v: k for k, v in val_generator.class_indices.items()}
print("\nРозподіл класів у валідаційному наборі:")
for i in range(len(unique_val_classes)):
    print(f"Клас {class_names_val[unique_val_classes[i]]}: {counts_val[i]} зображень")


Розподіл класів у навчальному наборі:
Клас random: 640 зображень
Клас toyota: 718 зображень

Розподіл класів у валідаційному наборі:
Клас random: 160 зображень
Клас toyota: 179 зображень


In [42]:
# Завантаження базової моделі Xception
base_model = Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False # Заморожуємо шари базової моделі спочатку

# Побудова нової моделі поверх базової
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x) # Додавання Dropout для регуляризації
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Компіляція моделі
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy', # Використовуємо binary_crossentropy для бінарної класифікації
    metrics=['accuracy']
)

# Колбеки
checkpoint = ModelCheckpoint(
    "best_" + MODEL_PATH,
    monitor='val_loss',
    save_best_only=True,
    mode='min'
)
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10, # Кількість епох без покращення для зупинки
    restore_best_weights=True
)

# Навчання моделі з колбеками
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=val_generator,
    validation_steps=val_generator.samples // 32,
    epochs=6,
    callbacks=[checkpoint, early_stopping]
)

Epoch 1/6
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step - accuracy: 0.5403 - loss: 0.6975

42/42 ━━━━━━━━━━━━━━━━━━━━ 25s 435ms/step - accuracy: 0.5416 - loss: 0.6968 - val_accuracy: 0.7531 - val_loss: 0.6112
Epoch 2/6
 1/42 ━━━━━━━━━━━━━━━━━━━━ 9s 227ms/step - accuracy: 0.6875 - loss: 0.6368

42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.6875 - loss: 0.6368 - val_accuracy: 0.7469 - val_loss: 0.6089
Epoch 3/6
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step - accuracy: 0.7500 - loss: 0.5948

42/42 ━━━━━━━━━━━━━━━━━━━━ 21s 495ms/step - accuracy: 0.7506 - loss: 0.5942 - val_accuracy: 0.8281 - val_loss: 0.5236
Epoch 4/6
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - accuracy: 0.8125 - loss: 0.5849 - val_accuracy: 0.8219 - val_loss: 0.5244
Epoch 5/6
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.8341 - loss: 0.5137

42/42 ━━━━━━━━━━━━━━━━━━━━ 14s 322ms/step - accuracy: 0.8348 - loss: 0.5132 - val_accuracy: 0.8844 - val_loss: 0.4504
Epoch 6/6
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.9375 - loss: 0.4570 - val_accuracy: 0.8875 - val_loss: 0.4538


In [43]:
loss, accuracy = model.evaluate(val_generator)
print(f"Accuracy: {accuracy:.4f}")

11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 392ms/step - accuracy: 0.8757 - loss: 0.4638
Accuracy: 0.8791


In [44]:
model.save(MODEL_PATH)

In [45]:
model = load_model(MODEL_PATH)

In [46]:
!cp "/content/drive/MyDrive/toyota.mp4" /content/

In [47]:
frame_size = (299, 299)
video_path = "/content/toyota.mp4"
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_num = 0
times_with_toyota = []
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Обробка кадру
    frame_num += 1
    image = cv2.resize(frame, frame_size)
    image = img_to_array(image)
    image = preprocess_input(image)
    image = np.expand_dims(image, axis=0)
    # Передбачення
    prediction = model.predict(image, verbose=0)[0][0]
    if prediction > 0.60:
        timestamp = frame_num / fps
        times_with_toyota.append(round(timestamp, 2))
cap.release()
# Результати
if times_with_toyota:
    print("Toyota logo detected at times (s):")
    print(times_with_toyota)
    print()
else:
    print("No Toyota logo detected.")


Toyota logo detected at times (s):
[0.03, 0.07, 0.1, 0.13, 0.17, 0.2, 0.23, 0.27, 0.3, 0.33, 0.37, 0.4, 0.43, 0.47, 0.5, 0.53, 0.57, 0.6, 0.63, 0.67, 0.7, 0.73, 0.77, 0.8, 0.83, 0.87, 0.9, 0.93, 0.97, 1.0, 1.03, 1.07, 1.1, 1.13, 1.17, 1.2, 1.23, 1.27, 1.3, 1.33, 1.37, 1.4, 1.43, 1.47, 1.5, 1.53, 1.57, 1.6, 1.63, 1.67, 1.7, 1.74, 1.77, 1.8, 1.84, 1.87, 1.9, 1.94, 1.97, 2.0, 2.04, 2.07, 2.1, 2.14, 2.17, 2.2, 2.24, 2.27, 2.3, 2.34, 2.37, 2.4, 2.44, 2.47, 2.5, 2.54, 2.57, 2.6, 2.64, 2.67, 2.7, 2.74, 2.77, 2.8, 2.84, 2.87, 2.9, 2.94, 2.97, 3.0, 3.04, 3.07, 3.1, 3.14, 3.17, 3.2, 3.24, 3.27, 6.74, 6.81, 6.84, 6.87, 6.91, 6.94, 6.97, 7.01, 7.04, 7.07, 7.11, 7.14, 7.24, 7.27, 7.34, 7.37, 7.41, 7.54, 7.57, 7.61, 7.67, 7.71, 7.74, 7.77, 7.84, 7.87, 7.91, 7.94, 7.97, 8.01, 8.04, 8.07, 8.11, 8.14, 8.17, 8.21, 8.24, 8.27, 8.31, 8.64, 9.31, 9.34, 9.38, 9.41, 9.51, 9.68, 9.71, 9.78, 9.81, 9.84, 9.88, 9.91, 9.98, 10.04, 10.08, 10.11, 10.14, 10.18, 10.24, 10.28, 10.31, 10.34, 10.38, 10.41, 10.44, 10.48,